# AI Response Evaluation Environment — GRPO RL Training

**Meta PyTorch OpenEnv Hackathon Finale**

This notebook trains `Qwen2.5-1.5B-Instruct` using GRPO (Group Relative Policy Optimization)
with the **AIResponseEvalEnv** as the reward source.

## What happens here
1. Load a local model with LoRA via Unsloth
2. Wrap AIResponseEvalEnv as a TRL-compatible tool environment
3. GRPO samples 4 candidate answers per prompt, scores via your graders
4. Model weights update toward higher-reward evaluation answers
5. Reward curve goes up — that is your learning signal
6. Save LoRA adapter + merged model

**Runtime:** T4 GPU (free Colab tier) — ~30-45 min for 300 steps

## Cell 1 — Install dependencies

In [ ]:
# Install Unsloth (optimised training) + TRL (GRPO) + OpenEnv
!pip install unsloth trl transformers datasets peft accelerate bitsandbytes -q
!pip install 'openenv-core[core]>=0.2.1' -q

# Clone your environment repo
import os
REPO_URL = 'https://huggingface.co/spaces/YOUR_USERNAME/ai_response_eval_env'
# OR use local path if running locally:
# !pip install -e /path/to/ai_response_eval_env
!pip install "openenv-code-assessment @ git+{REPO_URL}" -q

print('Installation complete')

## Cell 2 — Configuration

In [ ]:
import os

# Model — Qwen2.5-1.5B fits in T4 (15GB). Use 0.5B for free tier CPU.
MODEL_NAME  = 'Qwen/Qwen2.5-1.5B-Instruct'
ENV_URL     = 'http://localhost:7860'   # local server started in Cell 3
MAX_STEPS   = 300                        # increase to 500+ for better convergence
NUM_GEN     = 4                          # candidates per prompt (reduce to 2 if OOM)
LORA_RANK   = 16

# Your HuggingFace token for saving
HF_TOKEN = os.getenv('HF_TOKEN', '')    # set in Colab secrets or paste here
HF_REPO  = 'YOUR_USERNAME/code-assessment-grpo'  # where to push trained model

print(f'Model: {MODEL_NAME}')
print(f'Steps: {MAX_STEPS}')
print(f'HF token set: {bool(HF_TOKEN)}')

## Cell 3 — Start the environment server

In [ ]:
import subprocess, time, urllib.request, sys

proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '7860'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print('Waiting for server...')
for _ in range(30):
    try:
        if urllib.request.urlopen(f'{ENV_URL}/health', timeout=2).status == 200:
            print('Environment server ready!')
            break
    except:
        time.sleep(1)
else:
    print('Server timeout — check if ai_response_eval_env is installed')

## Cell 4 — Define the TRL environment wrapper

In [ ]:
import asyncio
from ai_response_eval_env import AIResponseEvalAction, AIResponseEvalEnv

class AIResponseEvalToolEnv:
    """
    TRL-compatible wrapper around AIResponseEvalEnv.
    The trainer discovers evaluate() as a tool and calls it automatically.
    """
    def __init__(self):
        self._env   = AIResponseEvalEnv(base_url=ENV_URL)
        self.reward = 0.0
        self._loop  = asyncio.new_event_loop()

    def _run(self, coro):
        return self._loop.run_until_complete(coro)

    def reset(self, **kwargs) -> str:
        """Start a new evaluation episode."""
        self.reward = 0.0
        result = self._run(self._env.reset())
        obs = result.observation
        return (
            f"TASK: {obs.task_type} | DIFFICULTY: {obs.difficulty}\n"
            f"INSTRUCTIONS: {obs.problem_description}\n"
            f"--- SCENARIO ---\n{obs.test_case_input}\n--- END SCENARIO ---"
        )

    def evaluate(self, answer: str) -> str:
        """
        Submit your evaluation of the AI response.

        Args:
            answer: Your judgment in the required format for the current task.

        Returns:
            Feedback on your evaluation and the next scenario.
        """
        result = self._run(self._env.step(AIResponseEvalAction(answer=answer)))
        obs    = result.observation
        self.reward += float(obs.reward or 0.0)
        if result.done or obs.done:
            return f'Episode done. Reward: {self.reward:.3f}. Feedback: {obs.feedback}'
        return (
            f'Feedback: {obs.feedback}\n'
            f'Next — TASK: {obs.task_type}\n'
            f'--- SCENARIO ---\n{obs.test_case_input}\n--- END SCENARIO ---'
        )


def reward_func(environments, **kwargs):
    """Read accumulated reward from each env instance after episode."""
    return [float(e.reward) for e in environments]

print('Environment wrapper defined')

## Cell 5 — Load model with Unsloth + LoRA

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 700

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,         # auto-detect bfloat16 on T4
    load_in_4bit=True,  # 4-bit quantisation — halves VRAM
)

# Apply LoRA — ONLY these adapter weights will be trained
# Everything else (base model) stays frozen
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print('LoRA adapters applied — only adapter weights will update during training')

## Cell 6 — Build dataset and train with GRPO

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
import json

# Dataset — simple prompt telling the model its role
# The actual task content is delivered by the environment via tool calls
SYSTEM_PROMPT = (
    'You are an expert AI response evaluator. '
    'You will be given evaluation scenarios and must call evaluate() '
    'with your judgment in the exact required format.'
)
dataset = Dataset.from_list([
    {"prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "Evaluate the scenario provided by the environment."},
    ]}
] * 200)  # 200 samples — GRPO cycles through these

# Reward log for plotting later
reward_log = []

class RewardLogCallback:
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'reward' in logs:
            entry = {'step': state.global_step, 'reward': logs['reward']}
            reward_log.append(entry)
            print(f"Step {state.global_step:4d} | reward={logs['reward']:.4f}")

# GRPO configuration — tuned for T4 (15GB VRAM)
training_args = GRPOConfig(
    num_generations=NUM_GEN,             # 4 candidates per prompt
    max_prompt_length=512,
    max_completion_length=150,
    temperature=0.9,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=MAX_STEPS,
    save_steps=100,
    logging_steps=5,
    output_dir='outputs/checkpoints',
    report_to='none',
    log_completions=True,
)

# Create GRPOTrainer with YOUR environment as reward source
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    args=training_args,
    train_dataset=dataset,
    environment_factory=AIResponseEvalToolEnv,  # <-- your env here
    callbacks=[RewardLogCallback()],
)

print(f'Starting GRPO training: {MAX_STEPS} steps')
print('The model will learn to evaluate AI responses by getting reward feedback...')
trainer.train()

## Cell 7 — Plot reward curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if reward_log:
    steps   = [r['step'] for r in reward_log]
    rewards = [r['reward'] for r in reward_log]

    # Smooth with moving average
    window  = max(2, len(rewards) // 10)
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

    fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0F1117')
    ax.set_facecolor('#1A1D27')
    ax.bar(steps, rewards, color='#4C9BE8', alpha=0.4, width=0.8, label='Raw reward')
    ax.plot(steps[window-1:], smoothed, color='#FFD700', linewidth=2.5, label='Smoothed')
    ax.set_title('GRPO Training — Reward Curve', color='white', fontsize=14, fontweight='bold')
    ax.set_xlabel('Training Step', color='#AAAAAA')
    ax.set_ylabel('Reward', color='#AAAAAA')
    ax.tick_params(colors='#AAAAAA')
    ax.legend(facecolor='#1A1D27', labelcolor='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')
    plt.tight_layout()
    plt.savefig('grpo_reward_curve.png', dpi=150, facecolor='#0F1117')
    plt.show()
    print(f'Curve saved to grpo_reward_curve.png')
    print(f'Initial reward: {rewards[0]:.4f}')
    print(f'Final reward:   {rewards[-1]:.4f}')
    print(f'Improvement:    {rewards[-1] - rewards[0]:+.4f}')
else:
    print('No reward data yet — run Cell 6 first')

## Cell 8 — Save weights

In [ ]:
import os

# Save LoRA adapter — small, for resuming training (~20-50MB)
print('Saving LoRA adapter...')
model.save_pretrained('outputs/lora_adapter')
tokenizer.save_pretrained('outputs/lora_adapter')
print('LoRA adapter saved to outputs/lora_adapter')

# Merge LoRA into base model and save full model (~3GB)
print('Merging LoRA into base model...')
model.save_pretrained_merged(
    'outputs/merged_model',
    tokenizer,
    save_method='merged_16bit',
)
print('Merged model saved to outputs/merged_model')

## Cell 9 — Push to HuggingFace Hub (optional)

In [ ]:
# Only run this if you want to publish the trained model
# Set HF_REPO to your desired model name above

if HF_TOKEN and HF_REPO != 'YOUR_USERNAME/code-assessment-grpo':
    print(f'Pushing to https://huggingface.co/{HF_REPO}...')
    model.push_to_hub_merged(
        HF_REPO,
        tokenizer,
        save_method='merged_16bit',
        token=HF_TOKEN,
    )
    print(f'Model live at: https://huggingface.co/{HF_REPO}')
else:
    print('Skipping Hub push — set HF_TOKEN and HF_REPO in Cell 2 to enable')

## Summary

After running all cells:

| Output | Location | Description |
|--------|----------|-------------|
| LoRA adapter | `outputs/lora_adapter/` | Small trained delta weights, for resuming |
| Merged model | `outputs/merged_model/` | Full model with LoRA merged in, for deployment |
| Reward curve | `grpo_reward_curve.png` | Training progress chart for demo |

**The reward going up in the chart is the proof that the agent learned from your environment.**